In [2]:
pip install streamlit

  Obtaining dependency information for streamlit from https://files.pythonhosted.org/packages/e4/08/c962e38f4450dafb98af49f8988a736dd67ae9cc26a5704c43269f506bc8/streamlit-1.42.2-py2.py3-none-any.whl.metadata
  Obtaining dependency information for altair<6,>=4.0 from https://files.pythonhosted.org/packages/aa/f3/0b6ced594e51cc95d8c1fc1640d3623770d01e4969d29c0bd09945fafefa/altair-5.5.0-py3-none-any.whl.metadata
  Obtaining dependency information for blinker<2,>=1.0.0 from https://files.pythonhosted.org/packages/10/cb/f2ad4230dc2eb1a74edf38f1a38b9b52277f75bef262d8908e60d957e13c/blinker-1.9.0-py3-none-any.whl.metadata
  Obtaining dependency information for cachetools<6,>=4.0 from https://files.pythonhosted.org/packages/72/76/20fa66124dbe6be5cafeb312ece67de6b61dd91a0247d1ea13db4ebb33c2/cachetools-5.5.2-py3-none-any.whl.metadata
  Obtaining dependency information for protobuf<6,>=3.20 from https://files.pythonhosted.org/packages/61/fa/aae8e10512b83de633f2646506a6d835b151edf4b30d18d73afd01447

In [3]:
import streamlit as st
import cv2
import numpy as np
import pandas as pd
import networkx as nx
from PIL import Image
import matplotlib.pyplot as plt
import re

In [4]:
import streamlit as st
import cv2
import numpy as np
import pytesseract
from PIL import Image
import io

def estimate_pixel_to_um(image):
    """
    解析圖片中的比例尺資訊（OCR + 傳統方法），無法識別則使用預設值。
    """
    default_pixel_to_um = 0.01
    img_array = np.array(image)
    img = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
    
    # OCR 讀取比例尺數值
    roi = img[-50:, :]  # 取圖片底部
    gray = cv2.cvtColor(roi, cv2.COLOR_BGR2GRAY)
    text = pytesseract.image_to_string(gray, config="--psm 6")
    
    # 嘗試解析 OCR 結果
    import re
    match = re.search(r"([\d.]+)\s*(µm|nm|mm)", text, re.IGNORECASE)
    if match:
        scale_bar_length = float(match.group(1))
        unit = match.group(2).lower()
        if unit == "nm":
            scale_bar_length /= 1000  # 轉換成 µm
        elif unit == "mm":
            scale_bar_length *= 1000  # 轉換成 µm
        return scale_bar_length / img.shape[1]  # 計算 µm/px
    
    print("❌ 無法解析比例尺，使用預設 1:1")
    return default_pixel_to_um


In [5]:
# 使用 Otsu + 形態學處理進行圖像分割
def segment_image_otsu(image):
    img_array = np.array(image.convert("L"))  # 轉換為灰度
    
    # Otsu 阈值法
    _, binary = cv2.threshold(img_array, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    
    # 形態學操作 - 去除噪聲
    kernel = np.ones((3,3), np.uint8)
    binary = cv2.morphologyEx(binary, cv2.MORPH_OPEN, kernel, iterations=2)
    binary = cv2.dilate(binary, kernel, iterations=1)
    
    # Canny 邊緣檢測
    edges = cv2.Canny(binary, 100, 200)
    return edges, binary


In [6]:
# 預處理圖像並檢測顆粒
def detect_particles(image, pixel_to_um):
    edges, binary = segment_image_otsu(image)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    particle_data = []
    centroids = []
    for contour in contours:
        M = cv2.moments(contour)
        if M["m00"] > 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            area_px = cv2.contourArea(contour)
            perimeter = cv2.arcLength(contour, True)
            circularity = 4 * np.pi * area_px / (perimeter ** 2) if perimeter > 0 else 0
            area_um = area_px * (pixel_to_um ** 2)
            centroids.append((cx, cy))
            particle_data.append((cx, cy, area_um, circularity))
    return centroids, particle_data, binary


In [7]:
import math  # 如果尚未引用 math 模組，請加上這一行

def detect_tpb_points_improved(centroids, pixel_to_um, threshold_um=0.5, angle_threshold=30):
    """
    改進版的 TPB 檢測函式：
      - 根據距離閥值建立鄰近關係。
      - 對每個與至少兩個鄰居連接的節點，計算從該節點指向每個鄰居的向量，
        並計算所有向量之間的夾角，以最小夾角作為可信度依據。
    
    參數：
      centroids: list of (x, y) 粒子質心
      pixel_to_um: 每個像素代表多少 µm
      threshold_um: 鄰近距離閥值，預設 0.5 µm
      angle_threshold: 角度過濾閥值，低於此值則可信度為 0（單位：度）
    
    回傳：
      tpb_points: 檢測到的 TPB 點列表
      confidence_scores: 每個 TPB 點對應的可信度分數（0～1）
    """
    threshold_px = threshold_um / pixel_to_um
    G = nx.Graph()
    
    # 建立鄰近關係，僅比較一次 (i < j)
    for i, (x1, y1) in enumerate(centroids):
        for j, (x2, y2) in enumerate(centroids):
            if i < j:
                distance = np.sqrt((x1 - x2) ** 2 + (y1 - y2) ** 2)
                if distance < threshold_px:
                    G.add_edge(i, j)
    
    tpb_points = []
    confidence_scores = []
    for node in G.nodes:
        neighbors = list(G[node])
        if len(neighbors) >= 2:
            x0, y0 = centroids[node]
            vectors = []
            for n in neighbors:
                xn, yn = centroids[n]
                dx, dy = xn - x0, yn - y0
                norm = math.hypot(dx, dy)
                if norm > 0:
                    vectors.append((dx / norm, dy / norm))
            if len(vectors) < 2:
                continue
            
            angles = []
            for i in range(len(vectors)):
                for j in range(i+1, len(vectors)):
                    dot = np.clip(vectors[i][0] * vectors[j][0] + vectors[i][1] * vectors[j][1], -1.0, 1.0)
                    angle = math.degrees(math.acos(dot))
                    angles.append(angle)
            
            if angles:
                min_angle = min(angles)
                if min_angle < angle_threshold:
                    conf = 0.0
                else:
                    conf = min_angle / 90.0
                    conf = max(0.0, min(conf, 1.0))
                
                tpb_points.append(centroids[node])
                confidence_scores.append(conf)
    
    return tpb_points, confidence_scores

In [8]:
# 可視化 TPB 結果
def visualize_tpb(image, centroids, tpb_points):
    img_array = np.array(image.convert("RGB"))  # 轉換為 RGB 格式
    img = cv2.cvtColor(img_array, cv2.COLOR_RGB2BGR)
    
    for (cx, cy) in centroids:
        cv2.circle(img, (cx, cy), 3, (0, 255, 0), -1)  # 綠色標記顆粒
    for (tx, ty) in tpb_points:
        cv2.circle(img, (tx, ty), 3, (0, 0, 255), -1)  # 紅色標記 TPB
    
    output_path = "tpb_output.png"
    cv2.imwrite(output_path, img)
    return output_path

In [9]:
# 計算 TPB 密度
def calculate_tpb_density(tpb_points, image_shape, pixel_to_um):
    img_area_um2 = (image_shape[0] * pixel_to_um) * (image_shape[1] * pixel_to_um)
    unit_count = img_area_um2 / 10
    tpb_density = len(tpb_points) / unit_count if unit_count > 0 else 0
    return tpb_density


In [10]:
# 保存數據到 CSV
def save_tpb_data(particle_data, tpb_points):
    df = pd.DataFrame(particle_data, columns=["Centroid_X", "Centroid_Y", "Area_um2", "Circularity"])
    df_tpb = pd.DataFrame(tpb_points, columns=["TPB_X", "TPB_Y"])
    df.to_csv("particles.csv", index=False)
    df_tpb.to_csv("tpb_points.csv", index=False)


In [11]:
# 繪製可信度直方圖
def plot_confidence_histogram(confidence_scores):
    """
    繪製候選 TPB 點可信度分布直方圖，並返回 matplotlib 的 Figure 對象。
    """
    import matplotlib.pyplot as plt  # 確保導入 matplotlib
    fig, ax = plt.subplots(figsize=(4, 3))
    ax.hist(confidence_scores, bins=10, range=(0, 1), edgecolor='black')
    ax.set_title("TPB 點可信度分布")
    ax.set_xlabel("可信度")
    ax.set_ylabel("點的數量")
    ax.set_xticks([0, 0.2, 0.4, 0.6, 0.8, 1.0])
    return fig


In [12]:
import streamlit as st
import time

def show_progress_overlay(duration=3000, tip_text="Tips: Use a clear, high-resolution image for better analysis results."):
    """
    在 Streamlit 顯示進度條，並提供提示資訊。
    duration: 顯示時長（毫秒）
    tip_text: 提示文字
    """
    progress_bar = st.progress(0)
    status_text = st.empty()
    
    for i in range(100):
        time.sleep(duration / 1000 / 100)  # 根據 duration 時長調整進度速度
        progress_bar.progress(i + 1)
    
    status_text.text(tip_text)
    time.sleep(1)
    progress_bar.empty()
    status_text.empty()

def hide_progress_bar():
    """ 清除進度條 """
    st.empty()

In [13]:
import streamlit as st
from PIL import Image
import matplotlib.pyplot as plt

# 初始化 Session State
if "page" not in st.session_state:
    st.session_state.page = 1

def next_page():
    st.session_state.page += 1

def prev_page():
    st.session_state.page -= 1

# **頁面 1：上傳圖片**
if st.session_state.page == 1:
    st.title("TPB Analysis - Upload Image")
    uploaded_file = st.file_uploader("Upload an image", type=["png", "jpg", "jpeg"])

    if uploaded_file:
        image = Image.open(uploaded_file)
        st.image(image, caption="Uploaded Image", use_column_width=True)
        st.session_state.image = image  # 存儲圖片供下一頁使用

    if st.button("Next"):
        next_page()

# **頁面 2：顯示結果**
elif st.session_state.page == 2:
    st.title("TPB Analysis - Results")

    if "image" in st.session_state:
        st.image(st.session_state.image, caption="Processed Image", use_column_width=True)

    fig, ax = plt.subplots(figsize=(4, 4))
    ax.hist([0.1, 0.3, 0.5, 0.7, 0.9], bins=10, range=(0, 1), edgecolor="black")
    ax.set_title("TPB confidence distribution")
    ax.set_xlabel("Confidence score")
    ax.set_ylabel("Number of TPB candidates")
    st.pyplot(fig)

    col1, col2 = st.columns(2)
    with col1:
        if st.button("Previous"):
            prev_page()
    with col2:
        if st.button("Next"):
            next_page()

# **頁面 3：顯示形態學分析**
elif st.session_state.page == 3:
    st.title("Morphology Analysis")

    if "image" in st.session_state:
        st.image(st.session_state.image, caption="Morphology Processed Image", use_column_width=True)

    st.write("Shape composition analysis will be displayed here.")

    col1, col2 = st.columns(2)
    with col1:
        if st.button("Previous"):
            prev_page()
    with col2:
        if st.button("Next"):
            next_page()

# **頁面 4：結束頁面**
elif st.session_state.page == 4:
    st.title("Final Page")
    st.write("(Empty Page)")

    if st.button("Restart"):
        st.session_state.page = 1


2025-02-24 12:42:31.778 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-24 12:42:31.779 WARNING streamlit.runtime.state.session_state_proxy: Session state does not function when running a script without `streamlit run`
2025-02-24 12:42:31.780 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-24 12:42:31.780 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-24 12:42:31.781 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2025-02-24 12:42:31.782 WARNING streamlit.runtime.scriptrunner_utils.script_run_c

In [14]:
import streamlit as st
from PIL import Image

def upload_image():
    """
    讓用戶上傳圖片，然後執行 TPB 分析。
    """
    uploaded_file = st.file_uploader("上傳圖片", type=["png", "jpg", "jpeg"])
    if uploaded_file is not None:
        image = Image.open(uploaded_file)
        st.session_state.image = image  # 存儲圖片供後續處理
        st.image(image, caption="已上傳圖片", use_column_width=True)
        st.success("圖片上傳成功！請繼續分析。")


In [15]:
def clean_ocr_text(ocr_text):
    """
    清理 OCR 讀取的文字，移除雜訊和特殊字符，確保格式統一
    """
    # 移除特殊字符和多餘空格
    cleaned_text = re.sub(r"[^0-9a-zA-Zµm]", "", ocr_text)
    return cleaned_text


In [16]:
def select_scale_region(image):
    """
    讓用戶選取比例尺區域，使用 OCR 讀取比例尺標示。
    """
    st.subheader("請選取比例尺區域")
    if "image" in st.session_state:
        st.image(st.session_state.image, caption="請選擇比例尺區域", use_column_width=True)
    
        # OCR 自動解析比例尺
        gray = cv2.cvtColor(np.array(st.session_state.image), cv2.COLOR_RGB2GRAY)
        ocr_text = pytesseract.image_to_string(gray, config="--psm 6")
        cleaned_text = clean_ocr_text(ocr_text)

        # 解析比例尺數值
        match = re.search(r"([\d.]+)\s*(µm|nm|mm)", cleaned_text)
        scale_length_um = None
        
        if match:
            scale_length_um = float(match.group(1))
            unit = match.group(2)
            if unit == "nm":
                scale_length_um /= 1000  # 轉換成 µm
            elif unit == "mm":
                scale_length_um *= 1000  # 轉換成 µm
            st.success(f"自動解析比例尺: {scale_length_um} µm")
        
        # 提供手動輸入選項
        scale_text = st.text_input("或手動輸入比例尺長度 (µm)", value=str(scale_length_um) if scale_length_um else "")

        if st.button("確定比例尺"):
            try:
                scale_length_um = float(scale_text)
                st.session_state.scale_length_um = scale_length_um
                st.success(f"比例尺設定成功: {scale_length_um} µm")
            except ValueError:
                st.error("請輸入有效的數字。")


In [17]:
def classify_shape(contour):
    """
    根據輪廓近似多邊形的頂點數和圓形度判斷顆粒的形狀。
    """
    epsilon = 0.03 * cv2.arcLength(contour, True)
    approx = cv2.approxPolyDP(contour, epsilon, True)
    if len(approx) == 3:
        return "Triangle"
    elif len(approx) == 4:
        return "Quadrilateral"
    elif len(approx) >= 5:
        area = cv2.contourArea(contour)
        perimeter = cv2.arcLength(contour, True)
        if perimeter == 0:
            return "Other"
        circularity = 4 * np.pi * area / (perimeter ** 2)
        if circularity > 0.7:
            return "Circle"
        else:
            return "Other"
    else:
        return "Other"

def compute_shape_ratios(binary_uint8, min_area=10):
    """
    偵測輪廓，並分類形狀，返回形狀數量和比例。
    """
    contours, _ = cv2.findContours(binary_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    shape_counts = {"Triangle": 0, "Quadrilateral": 0, "Circle": 0, "Other": 0}
    for cnt in contours:
        if cv2.contourArea(cnt) < min_area:
            continue
        shape = classify_shape(cnt)
        shape_counts[shape] += 1
    total = sum(shape_counts.values())
    ratios = {k: (v / total) * 100 if total > 0 else 0 for k, v in shape_counts.items()}
    return shape_counts, ratios

def format_shape_composition(ratios):
    """
    格式化形狀比例結果。
    """
    return ", ".join(f"{shape}: {perc:.1f}%" for shape, perc in ratios.items())

def annotate_image_with_shapes(orig_pil, binary_uint8):
    """
    在圖片上標註顆粒形狀。
    """
    orig_cv = cv2.cvtColor(np.array(orig_pil), cv2.COLOR_RGB2BGR)
    contours, _ = cv2.findContours(binary_uint8, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    for cnt in contours:
        shape = classify_shape(cnt)
        M = cv2.moments(cnt)
        if M["m00"] != 0:
            cx = int(M["m10"] / M["m00"])
            cy = int(M["m01"] / M["m00"])
            cv2.putText(orig_cv, shape[0], (cx, cy), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    return Image.fromarray(cv2.cvtColor(orig_cv, cv2.COLOR_BGR2RGB))

def plot_shape_composition_bar(ratios):
    """
    繪製形狀組成比例的柱狀圖。
    """
    import matplotlib.pyplot as plt
    fig, ax = plt.subplots(figsize=(5, 5))
    ax.bar(ratios.keys(), ratios.values(), color='skyblue', edgecolor='black')
    ax.set_title("Shape Composition", fontsize=14, fontweight='bold')
    ax.set_xlabel("Shape", fontsize=12)
    ax.set_ylabel("Percentage (%)", fontsize=12)
    ax.set_ylim(0, 100)
    for i, v in enumerate(ratios.values()):
        ax.text(i, v + 1, f"{v:.1f}%", ha='center', fontsize=10)
    return fig


In [ ]:
import streamlit as st
from PIL import Image
import re
import pytesseract
import cv2
import numpy as np

def upload_image():
    """
    讓用戶上傳圖片，然後執行 TPB 分析。
    """
    uploaded_file = st.file_uploader("上傳圖片", type=["png", "jpg", "jpeg"])
    if uploaded_file is not None:
        image = Image.open(uploaded_file)
        st.session_state.image = image  # 存儲圖片供後續處理
        st.image(image, caption="已上傳圖片", use_column_width=True)
        st.success("圖片上傳成功！請繼續分析。")

def show_user_guide():
    """顯示用戶指南"""
    st.sidebar.title("User Guide")
    st.sidebar.info(
        "1. 點擊 '上傳圖片' 選擇圖像文件 (JPG, PNG)。\n"
        "2. 系統將處理圖像並顯示: \n"
        "   - 原始圖像 \n"
        "   - 處理後圖像 \n"
        "   - TPB 可信度分布直方圖 \n"
        "3. 使用導航按鈕切換頁面: \n"
        "   - 頁面 2: TPB 分析結果 \n"
        "   - 頁面 3: 形態學分析 \n"
        "4. 確保使用清晰、高解析度的圖像以獲得最佳結果。\n"
        "5. 用戶指南按鈕始終可用於查看幫助資訊。"
    )

show_user_guide()
